# red neuronal para mnist

este proyecto implementa una red neuronal feedforward usando pytorch para clasificar digitos del dataset mnist.

## objetivos
- cargar dataset mnist
- dividir en train/validation/test
- entrenar al menos 3 modelos diferentes
- evaluar en conjunto de test

## importar librerias

solo se permiten pytorch y librerias auxiliares como matplotlib y tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, SubsetRandomSampler

import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
from tqdm import tqdm

print(f"pytorch version: {torch.__version__}")
print(f"torchvision version: {torchvision.__version__}")

## configuracion inicial

In [ ]:
# configurar device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"usando device: {device}")

# semilla para reproducibilidad
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print("configuracion lista")

## verificar instalacion

comprobar que pytorch funciona correctamente

In [ ]:
# crear tensor de prueba
x = torch.randn(3, 3)
print("tensor de prueba:")
print(x)
print(f"shape: {x.shape}")

# mover a device
x = x.to(device)
print(f"tensor en {x.device}")

print("\npytorch funcionando correctamente!")

## cargar dataset mnist

descargar y preparar el dataset mnist con las transformaciones basicas

In [ ]:
# transformaciones para el dataset
# normalizar con media y desviacion estandar de mnist
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# descargar dataset
train_dataset = torchvision.datasets.MNIST(root='./data', 
                                         train=True, 
                                         transform=transform, 
                                         download=True)

test_dataset = torchvision.datasets.MNIST(root='./data', 
                                        train=False, 
                                        transform=transform, 
                                        download=True)

print(f"dataset train: {len(train_dataset)} muestras")
print(f"dataset test: {len(test_dataset)} muestras")

## dividir train en entrenamiento y validacion

segun la actividad: 50,000 para entrenamiento, 10,000 para validacion

In [ ]:
# indices para dividir el dataset
train_size = 50000
val_size = 10000

# crear indices
indices = list(range(len(train_dataset)))
train_indices = indices[:train_size]
val_indices = indices[train_size:train_size + val_size]

print(f"entrenamiento: {len(train_indices)} muestras")
print(f"validacion: {len(val_indices)} muestras")
print(f"test: {len(test_dataset)} muestras")

# crear samplers para los dataloaders
train_sampler = SubsetRandomSampler(train_indices)
val_sampler = SubsetRandomSampler(val_indices)

## crear dataloaders

configurar dataloaders con batch size basico

In [ ]:
# configuracion de dataloaders
batch_size = 64

train_loader = DataLoader(train_dataset, 
                         batch_size=batch_size, 
                         sampler=train_sampler)

val_loader = DataLoader(train_dataset, 
                       batch_size=batch_size, 
                       sampler=val_sampler)

test_loader = DataLoader(test_dataset, 
                        batch_size=batch_size, 
                        shuffle=False)

print(f"train_loader: {len(train_loader)} batches")
print(f"val_loader: {len(val_loader)} batches")
print(f"test_loader: {len(test_loader)} batches")

## visualizar algunas muestras

mostrar ejemplos del dataset para verificar que esta bien cargado

In [ ]:
# obtener un batch de entrenamiento
dataiter = iter(train_loader)
images, labels = next(dataiter)

print(f"batch shape: {images.shape}")
print(f"labels shape: {labels.shape}")
print(f"labels: {labels[:10]}")

# visualizar primeras 8 imagenes
plt.figure(figsize=(12, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    # denormalizar para visualizar
    img = images[i].squeeze() * 0.3081 + 0.1307
    plt.imshow(img, cmap='gray')
    plt.title(f'digito: {labels[i].item()}')
    plt.axis('off')

plt.tight_layout()
plt.show()

print("\ndataset mnist cargado correctamente!")